# Supplementary figures — the `QTL_analysis.ipynb` group

| Panel | Written to | What it shows | Cost |
|---|---|---|---|
| `fig4_supfig1` | `plots/` | genomic-control λ in eQTL effect, u-sQTL vs p-sQTL, 49 tissues | seconds |
| `sup_fig_lambda` | `plots/` **and** `revision_plots/` | per-tissue QQ plot of eQTL p-values, 7 × 7 | hours |

`sup_fig_lambda` is saved twice in the source notebook — cells 104 and 105 are
identical apart from the destination directory — so it is built once here and
written to both.

Both share their data chain with main Figure 4, so this group reuses
`../Figure4/Figure4_helpers.py` rather than duplicating the permutation-pass and
eQTL nominal-pass readers.

---

### ⚠ A bug in `sup_fig_lambda`, reproduced on purpose

`QTL_analysis.ipynb` defines

```python
def get_var_eqtls(tissue, max_pvals=10000):
    ...
    nom_file = f'../code/results/eqtl/GTEx/Testis/cis_100000/nom/{chrom}.txt.gz'
```

The function takes a `tissue` argument and then ignores it — the path hard-codes
**Testis**. So the grey null distribution is the same Testis distribution in all
49 panels of the published figure, not each panel's own tissue.

The default here reproduces that, because it is what the paper shows. Pass
`use_source_tissue_bug=False` to read each panel's own tissue instead — that is
evidently what the code meant to do, and it will change the figure.

In [ ]:
import importlib
import os
import pickle

from matplotlib import pyplot as plt

import SupFig_QTL_plot_helpers
import SupFig_QTL_helpers
importlib.reload(SupFig_QTL_plot_helpers)
importlib.reload(SupFig_QTL_helpers)

from SupFig_QTL_helpers import (make_fig4_supfig1_data, make_sqtl_qq, prepare_qq,
                                PLOTS_DIR, REVISION_PLOTS_DIR)
from SupFig_QTL_plot_helpers import plot_fig4_supfig1, plot_sup_fig_lambda

for d in (PLOTS_DIR, REVISION_PLOTS_DIR, 'figure_data'):
    os.makedirs(d, exist_ok=True)


def save(name, out_dirs, dpi=300):
    for out_dir in ([out_dirs] if isinstance(out_dirs, str) else out_dirs):
        for ext in ('png', 'pdf', 'svg'):
            plt.savefig(f'{out_dir}/{name}.{ext}', dpi=dpi, bbox_inches='tight')
        print('wrote', f'{out_dir}/{name}.[png|pdf|svg]')

## `fig4_supfig1` — λ inflation per tissue

Cheap: reads only `code/analysis_files/sQTL_stats.tsv.gz`, which is itself now
produced by `rules/sqtl.smk :: MakesQTLStatsTable`.

In [ ]:
# fig4_supfig1 -- from QTL_analysis.ipynb
fig4_supfig1_data = make_fig4_supfig1_data()

plot_fig4_supfig1(fig4_supfig1_data)
save('fig4_supfig1', PLOTS_DIR)

# Original: plt.savefig('../code/plots/fig4_supfig1.png', dpi=300, bbox_inches='tight')

## `sup_fig_lambda` — per-tissue QQ plots

**This is the expensive cell in the supplementary set.** For each of 49 tissues
it runs one tabix query per significant sQTL and then reads whole eQTL nominal
chromosomes for the null. Budget hours, not minutes.

The result is pickled, so the plot cell below can be re-run from cache.

In [ ]:
# Fast path: load the cached QQ input instead of rebuilding it.
# Uncomment this cell and skip the next one.

# with open('figure_data/sqtl_qq.pickle', 'rb') as fh:
#     sqtl_qq = pickle.load(fh)

In [ ]:
# Builds sqtl_qq for all 49 tissues. SLOW -- see the note above.
# use_source_tissue_bug=True reproduces the published figure (Testis null in
# every panel); set it to False for the per-tissue null the code intended.

sqtl_qq = make_sqtl_qq(use_source_tissue_bug=True)

with open('figure_data/sqtl_qq.pickle', 'wb') as fh:
    pickle.dump(sqtl_qq, fh)
print(len(sqtl_qq), 'tissues')

In [ ]:
# sup_fig_lambda -- from QTL_analysis.ipynb
# Saved to both destinations: cells 104 and 105 of the source are the same plot,
# differing only in whether they write to revision_plots/ or plots/.

plot_sup_fig_lambda(sqtl_qq, prepare_qq)
save('sup_fig_lambda', [REVISION_PLOTS_DIR, PLOTS_DIR])

# Originals:
# plt.savefig('../code/revision_plots/sup_fig_lambda.png', dpi=300, bbox_inches='tight')
# plt.savefig('../code/plots/sup_fig_lambda.png', dpi=300, bbox_inches='tight')